# Falcon Eye: Data Ingestion Pipeline

<h3>Setup and Dynamic Link</h3>

In [ ]:
import pandas as pd


df_meast = pd.read_csv("./Middle-East.csv")
df_africa = pd.read_csv("./Africa.csv")


<h3>Merge Data</h3>

In [29]:

combined_df = pd.concat([df_africa, df_meast], ignore_index=True)

<h3>Filtering out non-MENA countries</h3>

In [ ]:

mena_countries = [
    "Algeria", "Bahrain", "Egypt", "Iran", "Iraq", "Israel", "Jordan", 
    "Kuwait", "Lebanon", "Libya", "Morocco", "Oman", "Palestine", 
    "Qatar", "Saudi Arabia", "Syria", "Tunisia", "United Arab Emirates", "Yemen"
]

# Filter the dataframe
df_mena = combined_df[combined_df['COUNTRY'].isin(mena_countries)]

print(df_mena.head())

<h3>Filtering out unwanted event types</h3>

In [ ]:

df_filtered = df_mena[df_mena['EVENT_TYPE'] != 'Sexual violence']

print(df_filtered['EVENT_TYPE'].unique())

<h3>Formatting, Date Conversion, and RowKey Generation</h3>

In [30]:
month_map = {
    'January': '01', 'February': '02', 'March': '03', 'April': '04',
    'May': '05', 'June': '06', 'July': '07', 'August': '08',
    'September': '09', 'October': '10', 'November': '11', 'December': '12'
}

def format_date_for_hbase(date_str):
    try:
        parts = str(date_str).split('-')
        day = parts[0].zfill(2)
        month =  month_map[parts[1]]
        year = parts[2]

        return f"{year}-{month}-{day}"
    except Exception:
        return date_str

combined_df['WEEK'] = combined_df['WEEK'].apply(format_date_for_hbase)


In [31]:
combined_df['row_key'] = combined_df['COUNTRY'] + "#" + combined_df['WEEK'] + "#" + combined_df['ID'].astype(str)


cols = ['row_key'] + [c for c in combined_df.columns if c != 'row_key']
combined_df = combined_df[cols]


combined_df.to_csv('main.csv', index=False)

<h3>Ingestion</h3>

In [ ]:
# from the root of the falcon eye project
# run this cmd to upload the csv file to the EC2 instance
# scp -i "falcon-key.pem" ./data/main.csv ubuntu@13.63.169.191:~/

# from the root of the project
# use this cmd to login to the EC2 instance
# ssh -i  "falcon-key.pem"  ubuntu@13.63.169.191


# next, inside the EC2 instance run this cmd
# to copy the file from EC2 into the Container
# docker cp /home/ubuntu/main.csv hdfs-namenode:/tmp/main.csv

# Now that the file is inside the container's /tmp folder
# put it into HDFS
# docker exec -it hdfs-namenode hdfs dfs -put -f /tmp/main.csv /main.csv

# check if the file is actually in HDFS
# docker exec -it hdfs-namenode hdfs dfs -ls /

# Run this command on EC2 terminal. This will import the csv file
# HBase will treat the first column as the HBASE_ROW_KEY and the rest as columns in the column family (cf).
# docker exec -it hbase-master hbase org.apache.hadoop.hbase.mapreduce.ImportTsv \
# -Dimporttsv.separator=',' \
# -Dimporttsv.skip.bad.lines=false \
# -Dimporttsv.skip.header=true \
# -Dimporttsv.columns=HBASE_ROW_KEY,cf:week,cf:region,cf:country,cf:admin1,cf:event_type,cf:sub_event_type,cf:events,cf:fatalities,cf:population_exposure,cf:disorder_type,cf:id,cf:latitude,cf:longitude \
# events hdfs://hdfs-namenode:8020/main.csv

<h3>Verification</h3>

In [ ]:
# start the thrift service in the EC2 session by using this cmd
# it will be accessible on /0.0.0.0:9090
# docker exec -it hbase-master hbase thrift start

In [9]:
import happybase

connection = happybase.Connection('13.63.169.191', port=9090)

try:
    connection.open()
    print("Connection Successful!")
except Exception as e:
    print(f"Failed to connect: {e}")

Connection Successful!


In [10]:
TABLE_NAME = "events"

try:
    table = connection.table(TABLE_NAME)
    for key, data in table.scan(limit=1):
        print(f"Sample Row Check:")
        print(f"RowKey: {key.decode()}")
        print(f"Sample Column (Country): {data.get(b'cf:country', b'N/A').decode()}")
    connection.close()
except Exception as e:
    print(f"Could not verify data: {e}")

Sample Row Check:
RowKey: Algeria#1996-12-28#83.0
Sample Column (Country): Algeria
